# Camada Bronze: Ingestão e Dados Brutos

In [0]:
%run ./00_Pipeline_Football_Setup


# Setup Inicial do Ambiente
## Arquitetura Agnóstica (Cloud vs Local)

Este projeto foi construído para ser **100% agnóstico de infraestrutura**. Isso significa que a mesma regra de negócio desenvolvida em PySpark pode ser executada sem nenhuma alteração em:

1. **Nuvem Corporativa (Ex: Databricks, AWS e Azure):** Usando o Unity Catalog, Glue.
2. **Ambiente Local (Docker / Jupyter):** Salvando os arquivos localmente usando o formato aberto Delta Lake (`.save()`).

A variável `ENVIRONMENT` controla essa chave de forma transparente para as camadas Bronze, Silver e Gold. Altere-a dependendo da sua stack.

Ambiente configurado: databricks


In [0]:
import requests, zipfile, io, shutil

fonte = {
    "repositorio": "worldcup.json",
    "url": "https://github.com/openfootball/worldcup.json/archive/refs/heads/master.zip"
}

pasta_raw = f"{PATH_RAW}/json_incremental/"
print(f"Baixando {fonte['repositorio']} da API...")

resposta = requests.get(fonte["url"], timeout=30)
with zipfile.ZipFile(io.BytesIO(resposta.content)) as zip_ref:
    pasta_destino = os.path.join(pasta_raw, fonte["repositorio"])
    os.makedirs(pasta_destino, exist_ok=True)
    zip_ref.extractall(pasta_destino)

if ENVIRONMENT == "databricks":
    volume_path = f"{PATH_BRONZE}/json_incremental/"
    for raiz, dirs, files in os.walk(pasta_raw):
        for arq in files:
            if arq.endswith(".json"):
                src = os.path.join(raiz, arq)
                rel = src.replace(os.path.normpath(pasta_raw), "").lstrip("\\/")
                dst = os.path.join(volume_path, rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copyfile(src, dst)
    caminho_leitura = volume_path
else:
    caminho_leitura = pasta_raw

print("Download concluido e pronto para leitura.")

Baixando worldcup.json da API...
Download concluido e pronto para leitura.


In [0]:
schema_gol = StructType([
    StructField("name", StringType(), True),
    StructField("minute", IntegerType(), True),
    StructField("offset", IntegerType(), True),
    StructField("penalty", BooleanType(), True),
    StructField("owngoal", BooleanType(), True)
])

schema_json = StructType([
    StructField("name", StringType(), True),
    StructField("matches", ArrayType(
        StructType([
            StructField("round", StringType(), True),
            StructField("date",  StringType(), True),
            StructField("time",  StringType(), True),
            StructField("team1", StringType(), True),
            StructField("team2", StringType(), True),
            StructField("group", StringType(), True),
            StructField("ground", StringType(), True),
            StructField("score", StructType([
                StructField("ft", ArrayType(IntegerType()), True),
                StructField("ht", ArrayType(IntegerType()), True),
                StructField("et", ArrayType(IntegerType()), True),
                StructField("p",  ArrayType(IntegerType()), True),
            ]), True),
            StructField("goals1", ArrayType(schema_gol), True),
            StructField("goals2", ArrayType(schema_gol), True)
        ])
    ), True)
])

In [0]:
print("Lendo arquivos JSON da API.")

df_bruto = (
    spark.read
    .schema(schema_json)
    .option("multiLine", "true")
    .option("recursiveFileLookup", "true")
    .json(caminho_leitura)
    .withColumn("source_file", F.col("_metadata.file_path"))
)


Lendo arquivos JSON da API.


In [0]:
# --- ANÁLISE EXPLORATÓRIA ---

print("\nCompetições encontradas na API ANTES do filtro:")
df_bruto.groupBy("name").count().orderBy("name").show(10, truncate=False)


Competições encontradas na API ANTES do filtro:
+----------------------+-----+
|name                  |count|
+----------------------+-----+
|Algeria               |2    |
|Argentina             |2    |
|Australia             |2    |
|Austria               |2    |
|Belgium               |2    |
|Bosnia & Herzegovina  |1    |
|Bosnia and Herzegovina|1    |
|Brazil                |2    |
|Canada                |2    |
|Cape Verde            |2    |
+----------------------+-----+
only showing top 10 rows


In [0]:
# --- ANÁLISE EXPLORATÓRIA ---

print("\nEdições da Copa do Mundo encontradas na API após o filtro")
df_teste_filtro = df_bruto.filter(
    F.col("name").contains("World Cup") & 
    ~F.col("name").contains("Club")
)
df_teste_filtro.select("name").distinct().orderBy("name").show(50, truncate=False)


Edições da Copa do Mundo encontradas na API após o filtro
+-------------------------+
|name                     |
+-------------------------+
|World Cup 1930           |
|World Cup 1934           |
|World Cup 1938           |
|World Cup 1950           |
|World Cup 1954           |
|World Cup 1958           |
|World Cup 1962           |
|World Cup 1966           |
|World Cup 1970           |
|World Cup 1974           |
|World Cup 1978           |
|World Cup 1982           |
|World Cup 1986           |
|World Cup 1990           |
|World Cup 1994           |
|World Cup 1998           |
|World Cup 2002           |
|World Cup 2006           |
|World Cup 2010           |
|World Cup 2014           |
|World Cup 2018           |
|World Cup 2022           |
|World Cup 2026           |
|World Cup 2026 Qualifying|
+-------------------------+



In [0]:
# Filtra apenas partidas da Copa do Mundo
# MOTIVO: A API worldcup.json traz dados misturados (ex: Mundial de Clubes e Eliminatorias).
# Esse filtro garante a qualidade da informacao, barrando times de clubes.
df_bruto = df_bruto.filter(
    F.col("name").contains("World Cup")
    & ~F.col("name").contains("Club")
    & F.col("matches").isNotNull()
)

df_partidas_expandidas = (
    df_bruto
    .select(
        F.col("name").alias("competition_name"),
        F.col("source_file"),
        F.explode("matches").alias("match")
    )
    .select(
        "competition_name", "source_file",
        F.col("match.round").alias("round"),
        F.col("match.date").cast("date").alias("match_date"),
        F.col("match.time").alias("match_time"),
        F.col("match.team1").alias("team_home"),
        F.col("match.team2").alias("team_away"),
        F.col("match.group").alias("group"),
        F.col("match.ground").alias("ground"),
        F.col("match.score.ft")[0].alias("score_ft_home"),
        F.col("match.score.ft")[1].alias("score_ft_away"),
        F.col("match.score.ht")[0].alias("score_ht_home"),
        F.col("match.score.ht")[1].alias("score_ht_away"),
        F.col("match.goals1").alias("goals1"),
        F.col("match.goals2").alias("goals2")
    )
    .withColumn("match_id", F.md5(
        F.concat_ws("_", F.col("match_date").cast("string"), "team_home", "team_away")
    ))
    .withColumn("ingested_at", F.current_timestamp())
)

df_partidas_bruto = df_partidas_expandidas.drop("goals1", "goals2").filter(F.col("score_ft_home").isNotNull() & F.col("score_ft_away").isNotNull())

print(f"Total de partidas processadas: {df_partidas_bruto.count()}")

Total de partidas processadas: 1066


### Frequência de Atualização e Carga Incremental

Para atender ao requisito de **Sustentabilidade e Governança**, o agendamento (Schedule) deste pipeline deve seguir o calendário oficial do negócio (FIFA):

- **Fora de época (Maior parte do tempo):** O pipeline não precisa rodar. Os dados históricos estão congelados.
- **Ano de Copa do Mundo (A cada 4 anos):** Durante o mês do torneio, este notebook é orquestrado para rodar **diariamente**.

**Como a Carga Incremental funciona aqui:**
Graças ao comando `MERGE` do Delta Lake abaixo, não apagamos o histórico de 1930 a 2022. O código compara o JSON da API com a tabela Delta:
1. Jogo novo que acabou de acontecer? **INSERT**
2. Jogo que já existe, mas o placar mudou? **UPDATE**
3. Jogo antigo intacto? **IGNORA** (economia de custo/compute).

In [0]:
# Carga Incremental da Tabela de Partidas (matches)
if ENVIRONMENT == "databricks":
    tabela_existe = spark.catalog.tableExists(f"{CATALOG_BRONZE}.matches")
else:
    import os
    tabela_existe = os.path.exists(f"{PATH_BRONZE}/matches/_delta_log")

if not tabela_existe:
    print("Criando tabela bronze.matches pela primeira vez...")
    if ENVIRONMENT == "databricks":
        df_partidas_bruto.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG_BRONZE}.matches")
    else:
        df_partidas_bruto.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{PATH_BRONZE}/matches")
else:
    print("Executando MERGE na tabela bronze.matches...")
    if ENVIRONMENT == "databricks":
        tabela_delta = DeltaTable.forName(spark, f"{CATALOG_BRONZE}.matches")
    else:
        tabela_delta = DeltaTable.forPath(spark, f"{PATH_BRONZE}/matches")

    (tabela_delta.alias("historico")
        .merge(
            df_partidas_bruto.alias("novo"),
            "historico.match_id = novo.match_id"
        )
        .whenMatchedUpdate(set={
            "score_ft_home": "novo.score_ft_home",
            "score_ft_away": "novo.score_ft_away",
            "score_ht_home": "novo.score_ht_home",
            "score_ht_away": "novo.score_ht_away",
            "ingested_at":   "novo.ingested_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    
    # --- OBSERVABILIDADE (LOGGING) ---
    print("\n RESULTADO DO MERGE: Carga Incremental")
    # Pega a ultima operacao realizada na tabela (que foi o nosso merge)
    metricas = tabela_delta.history(1).select("operationMetrics").collect()[0]["operationMetrics"]
    print(f"-> Linhas Inseridas (Novos Jogos): {metricas.get('numTargetRowsInserted', 0)}")
    print(f"-> Linhas Atualizadas (Placares Corrigidos): {metricas.get('numTargetRowsUpdated', 0)}")

Criando tabela bronze.matches pela primeira vez...


In [0]:
# Extracao de Gols
df_partidas_com_gols = df_partidas_expandidas.filter(
    (F.col("goals1").isNotNull() & (F.size("goals1") > 0))
    | (F.col("goals2").isNotNull() & (F.size("goals2") > 0))
)

df_gols_mandante = (
    df_partidas_com_gols
    .filter(F.col("goals1").isNotNull() & (F.size("goals1") > 0))
    .select(
        "match_id", "competition_name", "match_date",
        "team_home", "team_away", "score_ft_home", "score_ft_away",
        "source_file",
        F.posexplode("goals1").alias("pos", "goal")
    )
    .select("*",
        F.col("team_home").alias("team"),
        F.col("team_away").alias("opponent"),
        F.col("goal.name").alias("scorer"),
        F.col("goal.minute").alias("minute"),
        F.col("goal.penalty").cast("int").alias("is_penalty"),
        F.col("goal.owngoal").cast("int").alias("is_own_goal"),
    ).drop("goal")
)

df_gols_visitante = (
    df_partidas_com_gols
    .filter(F.col("goals2").isNotNull() & (F.size("goals2") > 0))
    .select(
        "match_id", "competition_name", "match_date",
        "team_home", "team_away", "score_ft_home", "score_ft_away",
        "source_file",
        F.posexplode("goals2").alias("pos", "goal")
    )
    .select("*",
        F.col("team_away").alias("team"),
        F.col("team_home").alias("opponent"),
        F.col("goal.name").alias("scorer"),
        F.col("goal.minute").alias("minute"),
        F.col("goal.penalty").cast("int").alias("is_penalty"),
        F.col("goal.owngoal").cast("int").alias("is_own_goal"),
    ).drop("goal")
)

df_gols_bruto = (
    df_gols_mandante.unionByName(df_gols_visitante)
    .fillna(0, subset=["is_penalty", "is_own_goal"])
    .withColumn("ingested_at", F.current_timestamp())
)

window_spec = Window.partitionBy("match_id").orderBy("minute", "pos")
df_gols_bruto = df_gols_bruto.withColumn("event_seq", F.row_number().over(window_spec)).drop("pos")

print("Salvando bronze.goals_raw...")
if ENVIRONMENT == "databricks":
    df_gols_bruto.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG_BRONZE}.goals_raw")
else:
    df_gols_bruto.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{PATH_BRONZE}/goals_raw")

Salvando bronze.goals_raw...


In [0]:
%sql

select * from workspace.bronze.goals_raw
limit 5

match_id,competition_name,match_date,team_home,team_away,score_ft_home,score_ft_away,source_file,team,opponent,scorer,minute,is_penalty,is_own_goal,ingested_at,event_seq
0030aa35f001c27af1361f0158af3590,World Cup 2018,2018-07-07,Sweden,England,0,2,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2018/worldcup.json,England,Sweden,Harry Maguire,30,0,0,2026-07-17T00:10:17.356Z,1
0030aa35f001c27af1361f0158af3590,World Cup 2018,2018-07-07,Sweden,England,0,2,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2018/worldcup.json,England,Sweden,Dele Alli,58,0,0,2026-07-17T00:10:17.356Z,2
0040274278df02c89b2689b49bcc3bfa,World Cup 2018,2018-07-06,Uruguay,France,0,2,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2018/worldcup.json,France,Uruguay,Raphaël Varane,40,0,0,2026-07-17T00:10:17.356Z,1
0040274278df02c89b2689b49bcc3bfa,World Cup 2018,2018-07-06,Uruguay,France,0,2,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2018/worldcup.json,France,Uruguay,Antoine Griezmann,61,0,0,2026-07-17T00:10:17.356Z,2
00bec281fafc7d47075d14d419616860,World Cup 1930,1930-07-17,Yugoslavia,Bolivia,4,0,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/1930/worldcup.json,Yugoslavia,Bolivia,Bek,60,0,0,2026-07-17T00:10:17.356Z,1


In [0]:
%sql

select * from workspace.bronze.matches
limit 5 

competition_name,source_file,round,match_date,match_time,team_home,team_away,group,ground,score_ft_home,score_ft_away,score_ht_home,score_ht_away,match_id,ingested_at
World Cup 1930,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/1930/worldcup.json,Matchday 1,1930-07-13,null,France,Mexico,Group 1,"Estadio Pocitos, Montevideo",4,1,3,0,1eda0860500cb7f1848fbd1e3d1c4e0d,2026-07-17T00:10:12.833Z
World Cup 1930,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/1930/worldcup.json,Matchday 3,1930-07-15,null,Argentina,France,Group 1,"Estadio Parque Central, Montevideo",1,0,0,0,3b6e7a5527f7c35ff406c58adf9ed63b,2026-07-17T00:10:12.833Z
World Cup 1930,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/1930/worldcup.json,Matchday 4,1930-07-16,null,Chile,Mexico,Group 1,"Estadio Parque Central, Montevideo",3,0,1,0,aaed4ce53e8c205b3beaa5291590abc9,2026-07-17T00:10:12.833Z
World Cup 1930,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/1930/worldcup.json,Matchday 7,1930-07-19,null,Chile,France,Group 1,"Estadio Centenario, Montevideo",1,0,0,0,8ddf3688b8ba66ae57758461b19f267f,2026-07-17T00:10:12.833Z
World Cup 1930,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/1930/worldcup.json,Matchday 7,1930-07-19,null,Argentina,Mexico,Group 1,"Estadio Centenario, Montevideo",6,3,3,1,9c56f41e9c1389df6b49fe243f9c2923,2026-07-17T00:10:12.833Z
